In [2]:
import os

DATA_DIR = os.path.join('..', '..', 'data')
print(os.listdir(DATA_DIR))

['experiment', 'test', 'train', 'valid']


In [23]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms as T

In [16]:
image_width = 256
image_height = 128
batch_size = 128
stats = (0.06806663, 0.06942986, 0.07024706), (0.03520789, 0.03602721, 0.03545172)

# Building the Model

In [358]:
class DownSample(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=4):
        super(DownSample, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, 
                      stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2)
        )
        
    def forward(self, x):
        x = self.model(x)
        return x

class UpSample(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=4):
        super(UpSample, self).__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=kernel_size, 
                               stride=2, padding=1, bias=False),
            nn.InstanceNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip_input):
        x = self.model(x)
        x = torch.cat((x, skip_input), 1)
        return x

### Generator

In [364]:
class Generator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3):
        super(Generator, self).__init__()

        # Encoder (Downsampling)
        self.down1 = DownSample(in_channels, 64)  # -> (64, 64, 128)
        self.down2 = DownSample(64, 128)          # -> (128, 32, 64)
        self.down3 = DownSample(128, 256)         # -> (256, 16, 32)
        self.down4 = DownSample(256, 512)         # -> (512, 8, 16)
        self.down5 = DownSample(512, 1024)        # -> (1024, 4, 8)
        self.down6 = DownSample(1024, 1024)       # -> (1024, 2, 4)
        self.down7 = DownSample(1024, 1024)       # -> (1024, 1, 2)

        # Decoder (Upsampling)
        self.up1 = UpSample(1024, 1024)           # -> (1024, 2, 4)
        self.up2 = UpSample(2048, 1024)           # -> (1024, 4, 8)
        self.up3 = UpSample(2048, 512)            # -> (512, 8, 16)
        self.up4 = UpSample(1024, 256)            # -> (256, 16, 32)
        self.up5 = UpSample(512, 128)             # -> (128, 32, 64)
        self.up6 = UpSample(256, 64)              # -> (64, 64, 128)

        self.final = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.ZeroPad2d((1, 0, 1, 0)),
            nn.Conv2d(128, out_channels, 4, padding=1), # -> (64, 128, 256)
            nn.Tanh()
        )

    def forward(self, x):
        # U-Net generator with skip connections from encoder to decoder
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        d4 = self.down4(d3)
        d5 = self.down5(d4)
        d6 = self.down6(d5)
        d7 = self.down7(d6)
        
        u1 = self.up1(d7, d6)
        u2 = self.up2(u1, d5)
        u3 = self.up3(u2, d4)
        u4 = self.up4(u3, d3)
        u5 = self.up5(u4, d2)
        u6 = self.up6(u5, d1)
        u7 = self.final(u6)
        return u7

In [370]:
image = torch.rand((1, 3, 128, 256))
out_channels = 3
generator = Generator()
output = generator(image)
print(output.shape)

torch.Size([1, 3, 128, 256])


### Discriminator

In [378]:
def conv_block(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=bias),
        nn.LeakyReLU(0.2, inplace=True)
    )

In [391]:
class Discriminator(nn.Module):
    def __init__(self, in_channels=6, kernel_size=4):
        super(Discriminator, self).__init__()

        self.model = nn.Sequential(
            conv_block(in_channels, 64), # 128x256
            conv_block(64, 128), # 64x128
            conv_block(128, 256), # 32x64
            conv_block(256, 512), # 16x32
            nn.ZeroPad2d((1, 0, 1, 0)),
            nn.Conv2d(512, 1, 4, padding=1, bias=False) # 8x16
        )

    def forward(self, image1, image2):
        img_input = torch.cat((image1, image2), 1)
        return self.model(img_input)

In [393]:
image1 = torch.rand((1, 3, 128, 256))
image2 = torch.rand((1, 3, 128, 256))

out_channels = 3
discriminator = Discriminator()
output = discriminator(image1, image2)
print(output.shape)

torch.Size([1, 1, 8, 16])
